### Generate SMILES with minGPT

In [ ]:
import scipy.stats as stats

if not hasattr(stats, "gibrat"):
    from scipy.stats import lognorm
    stats.gibrat = lognorm

In [14]:
import os
from openai import OpenAI

OPENAI_API_KEY = os.environ["OPENAI_API_KEY"]
client = OpenAI(api_key=OPENAI_API_KEY)

In [ ]:
# Give GPT-4o a prompt and automatically parse the output into a SMILES list.
# It can generate an arbitrary number of SMILES (although GPT-4o has a per-call token limit).
def generate_gpt4o_smiles(prompt, n=30, model="gpt-4o-mini"):
    """
    Generate SMILES using GPT-4o/Mini with a few-shot prompt.
    Returns a list of SMILES strings.
    """
    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        temperature=1.0,
        max_tokens=2000  # Allow longer outputs
    )

    # Compatible with openai v1 SDK: content is an attribute, not a dict
    text = response.choices[0].message.content

    # One SMILES per line; remove empty lines and odd prefixes
    smiles = []
    for line in text.split("\n"):
        line = line.strip()
        if not line:
            continue
        # Remove possible leading numbering like "1." "2."
        if line[0].isdigit() and line[1] == ".":
            line = line[2:].strip()
        smiles.append(line)

    # Limit to n outputs (GPT may generate more than requested)
    return smiles[:n]

In [ ]:
# Prepare a few-shot high-conductivity prompt
# Generate high-conductivity SMILES
high_prompt_gpt4o = """
You are a polymer electrolyte design assistant. 
Generate 100 unique polymer-like molecules in SMILES format with HIGH ionic conductivity.

Requirements:
- Each molecule must contain exactly one [Cu] and one [Au].
- Include polar groups (O, N, CO).
- Must be valid SMILES, one per line.
- Must not repeat the examples.

Examples:
NC(=O)CSCC(CO[Cu])OC(=O)[Au]
CCN(CCO[Cu])CCC(CO)OC(=O)[Au]
OCNCCC(COC[Cu])COC(=O)[Au]

Now generate 100 new high-conductivity polymer molecules.
Return ONLY SMILES lines, nothing else.
"""

gpt4o_high_results = generate_gpt4o_smiles(high_prompt_gpt4o, n=100)
gpt4o_high_results[:10]


['NCOC(=O)C(CO[Cu])NCC(=O)[Au]',
 'O=C(OC[Au])CCN(CCO[Cu])C(=O)C',
 'C(CO)N(CC[Cu])C(=O)OCC(=O)[Au]',
 'OCN(COC[Au])C(CO[Cu])CC(=O)',
 'COC(=O)C(CCN[Cu])OC(=O)[Au]',
 'N(CO[Cu])C(=O)C(CO[Au])CCOC',
 'C(NCCO[Cu])C(=O)OC(=O)[Au]',
 'O=C(O[Cu])CCN(CCO)C(=O)[Au]',
 'NCOC[Cu]C(=O)CC(CCOC)[Au]',
 '10. CC(=O)O[Cu]CNC(=O)CCO[Au]']

In [ ]:
# Prepare a few-shot low-conductivity prompt
# Generate low-conductivity SMILES
low_prompt_gpt4o = """
You are a SMILES generator for polymer electrolytes.

Task:
Generate 100 VALID SMILES strings that have LOW ionic conductivity.

Hard constraints (must ALL be satisfied):
1. Each SMILES must contain EXACTLY one [Cu] and one [Au].
2. The backbone must be HYDROPHOBIC:
   - Use mainly C and H atoms.
   - MAY include at most ONE oxygen (O) OR ONE nitrogen (N), but not both.
3. MUST NOT contain more than one polar group (=O or -O- or -N-).
4. No whitespace, no numbering, no comments.
5. One SMILES per line.
6. Return ONLY SMILES lines.

Soft constraints (increase likelihood of low conductivity):
- Prefer long aliphatic carbon chains.
- Keep heteroatoms near the chain ends.

Examples (valid patterns):
CCCCCCC(C)CCO[Cu]CCC[Au]
CCC(C)CCCCCCC(=O)C[Cu]C[Au]
CCCCCCCCCN[Cu]CCCC(=O)[Au]

Now generate 100 new low-conductivity polymer molecules.
Return ONLY SMILES, no explanations.

"""

gpt4o_low_results = generate_gpt4o_smiles(low_prompt_gpt4o, n=100)
gpt4o_low_results[:10]

['CCCCCCCCCCCCCCCCCCC[Cu]CCCCCCCCCCCCC[Au]',
 'CCCCCCCCCCCCCCCCCC(C)[Cu]CCCCCCCCCCCC[Au]',
 'CCCCCCCCCCCCCCCCC[Cu]CCCCCCCCCCCCCCCCA[Au]',
 'CCCCCCCCCCCCCCCCC[Cu]CCCCCCCCCCC(=O)CC[Au]',
 'CCCCCCCCCCCCC(C)CC[Cu]CCCCCCCC(=O)C[Au]',
 'CCCCCCCCCCCCC(C)CC[Cu]CCCCCCCC(=O)[Au]',
 'CCCCCCCCCCCCC(C)CC[Cu]CCCCCCCCC[Au]',
 'CCCCCCCCCCCCCCCC[Cu]CCCCCCCCCCC(C)C[Au]',
 'CCCCCCCCCCCCCCCC[Cu]CCCCCCCCCCC(=O)CC[Au]',
 'CCCCCCCCCCCCCCC(C)CC[Cu]CCCCCCCCCCC[Au]']

### Post processing

In [ ]:
import re

def clean_gpt4o_output(raw_list):
    cleaned = []
    
    for line in raw_list:
        line = line.strip()
        if not line:
            continue

        # 1) Remove leading numbering, e.g., "10. ", "3. "
        #    Pattern: optional leading spaces + digits + dot + optional spaces
        line = re.sub(r'^\s*\d+\.\s*', '', line)

        # 2) Must contain both Cu and Au (this rule is kept)
        if "[Cu]" not in line or "[Au]" not in line:
            continue

        # 3) Filter out lines that are clearly not SMILES (contain long English words)
        #    Since the prompt already specifies “output SMILES only”,
        #    this case should now be rare
        if re.search(r"[A-Za-z]{3,}", line) and not re.search(r"[A-Z][a-z]?", line):
            continue

        # 4) Disallow spaces, commas, semicolons, etc.
        if re.search(r"[ ,:;]", line):
            continue

        # 5) Basic character check: only allow atoms, digits, brackets, and bond symbols
        if not re.match(r"^[A-Za-z0-9\[\]\(\)=#]+$", line):
            continue

        cleaned.append(line)

    return cleaned


gpt4o_high_clean = clean_gpt4o_output(gpt4o_high_results)
gpt4o_low_clean  = clean_gpt4o_output(gpt4o_low_results)
len(gpt4o_high_clean)
len(gpt4o_high_clean), len(gpt4o_low_clean)


98

In [ ]:
import json

# saving high conductivity
with open("../../data/generated/gpt4o/gpt4o_high_clean.json", "w") as f:
    json.dump(gpt4o_high_clean, f)

# saving low conductivity
with open("../../data/generated/gpt4o/gpt4o_low_clean.json", "w") as f:
    json.dump(gpt4o_low_clean, f)

print("Saved!")

Saved!
